<a href="https://colab.research.google.com/github/alexander-toschev/cv-course/blob/main/Tasks/Task3_NSFW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Optimizing F1 Threshold on CIFAR-10 with Noise Examples (Frog as NSFW)

In [ ]:
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
!pip install --upgrade gspread pandas google-auth
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from IPython.display import display
import random
# Authenticate and create the PyDrive client.
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.3/212.3 kB 14.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.38.0
    Uninstalling google-auth-2.38.0:
      Successfully uninstalled google-auth-2.38.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.39.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
# FILL THIS
student_name = "ELON MUSK"
group_id = "11-101"

In [ ]:
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1VIp1PdTVfR4rWP44YYWzz58hVf-_wLpX4NwXmcG7MNo/edit?usp=sharing"
sh = gc.open_by_url(SPREADSHEET_URL)
worksheet = sh.sheet1
score = 0
# Ensure header row exists
if not worksheet.get_all_values():
    worksheet.append_row(["Student Name", "Group","TaskID", "Score"])

In [ ]:
# MAIN NOTEBOOK GOES HERE
# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
task_id = "Task3_NSFW"
score = 0
max_score = 15

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, precision_recall_curve, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from tensorflow.keras.datasets import cifar10
import matplotlib.animation as animation
from IPython.display import HTML

In [2]:
(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()
X_full = np.concatenate([X_train_full, X_test_full])
y_full = np.concatenate([y_train_full, y_test_full]).flatten()

mask = np.isin(y_full, [3, 5, 6])  # cat, dog, frog (noise)
X = X_full[mask]
y_raw = y_full[mask]
y = np.where(y_raw == 3, 0, 1)  # cat=0, others=1

X_flat = X.reshape(X.shape[0], -1) / 255.0
X_flat, y = shuffle(X_flat, y, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_flat, y, test_size=0.2, random_state=42)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [3]:
model = LogisticRegression(max_iter=600)
model.fit(X_train, y_train)
y_scores = model.predict_proba(X_val)[:, 1]

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
precision, recall, thresholds = precision_recall_curve(y_val, y_scores)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_final = (y_scores > best_thresh).astype(int)

print(f"Best threshold: {best_thresh:.2f}")

Best threshold: 0.07


In [ ]:
def test_model_predictions():
    global score
    score = 0
    acc = accuracy_score(y_val, y_pred_final)
    f1 = f1_score(y_val, y_pred_final)
    assert acc > 0.6, f"Accuracy too low: {acc:.2f}"
    score+=5
    assert f1 > 0.8, f"F1 score too low: {f1:.2f}"
    score+=5
    assert 0.2 < best_thresh < 0.8, f"Suspicious threshold: {best_thresh:.2f}"
    score+=5

    print("✅ All tests passed.")

test_model_predictions()

AssertionError: Suspicious threshold: 0.06

In [ ]:

# DO NOT MODIFY  !!!
# DO NOT EXECUTE !!!
# Save the result to Google Sheets
from datetime import datetime

# Get current date and time
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d %H:%M:%S")
worksheet.append_row([student_name,group_id, task_id, score, timestamp])

print(f"Test completed! {student_name}, your score is {score}/{max_score}.")